In [1]:
import asyncio, websockets, json

async def main():
    url = "ws://localhost:8080/ws/realtime?api_key=sk_live_3jf83jf83jf"
    async with websockets.connect(url) as ws:
        await ws.send(json.dumps({"type": "input_text", "text": "Hello, can you speak in Urdu?"}))
        while True:
            msg = await ws.recv()
            print(msg)

asyncio.run(main())


RuntimeError: asyncio.run() cannot be called from a running event loop

In [2]:
import nest_asyncio
import asyncio
import websockets
import json

nest_asyncio.apply()  # allow nested event loops

async def main():
    url = "ws://localhost:8080/ws/realtime?api_key=sk_live_3jf83jf83jf"
    async with websockets.connect(url) as ws:
        await ws.send(json.dumps({"type": "input_text", "text": "Hello, can you speak in Urdu?"}))
        while True:
            msg = await ws.recv()
            print(msg)

# Instead of asyncio.run(), use:
asyncio.get_event_loop().run_until_complete(main())


{"type":"state_change","state":"LISTENING","timestamp":1770989253.7488596}


KeyboardInterrupt: 

{"type":"error","message":"Session idle timeout","timestamp":1770989554.0066955}


In [3]:
import nest_asyncio
import asyncio
import websockets
import json

nest_asyncio.apply()  # for Jupyter

async def main():
    url = "ws://localhost:8080/ws/realtime?api_key=sk_live_3jf83jf83jf"
    async with websockets.connect(url) as ws:
        # Trigger text input
        await ws.send(json.dumps({
            "type": "input_text",
            "text": "Hello, can you speak in Urdu?"
        }))

        while True:
            msg = await ws.recv()
            data = json.loads(msg)
            print(json.dumps(data, indent=2, ensure_ascii=False))

asyncio.get_event_loop().run_until_complete(main())


{
  "type": "state_change",
  "state": "LISTENING",
  "timestamp": 1770989381.8782535
}


KeyboardInterrupt: 

{
  "type": "error",
  "message": "Session idle timeout",
  "timestamp": 1770989682.1200836
}


In [ ]:
import base64
import numpy as np
import soundfile as sf
import json
import websockets
import asyncio

async def send_audio(uri, api_key, filepath):
    audio, sr = sf.read(filepath)
    assert sr == 16000, f"Sample rate must be 16kHz, got {sr}"
    if audio.ndim > 1:  # convert to mono
        audio = audio.mean(axis=1)
    # Convert float32 [-1,1] to int16
    if audio.dtype != np.int16:
        audio = (audio * 32767).astype(np.int16)
    chunk_size = 3200  # 100ms chunks at 16kHz mono
    async with websockets.connect(uri, extra_headers={"X-API-Key": api_key}) as ws:
        for i in range(0, len(audio), chunk_size):
            chunk = audio[i:i+chunk_size].tobytes()
            b64 = base64.b64encode(chunk).decode("ascii")
            await ws.send(json.dumps({"type": "input_audio_buffer.append", "audio": b64}))
        await ws.send(json.dumps({"type": "input_audio_buffer.commit"}))
        await ws.send(json.dumps({"type": "input_audio_buffer.flush"}))
        # Read server response
        while True:
            msg = await ws.recv()
            print(msg)

asyncio.run(send_audio(
    "ws://localhost:8080/ws/realtime",
    "sk_live_3jf83jf83jf
    "output.pcm"
))


In [5]:
pip install soundfile


  Using cached soundfile-0.13.1-py2.py3-none-manylinux_2_28_x86_64.whl.metadata (16 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached soundfile-0.13.1-py2.py3-none-manylinux_2_28_x86_64.whl (1.3 MB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [soundfile]━━━━━━━━ 2/3 [soundfile]
Note: you may need to restart the kernel to use updated packages.


In [6]:
import base64
import numpy as np
import soundfile as sf
import json
import websockets
import asyncio

async def send_audio(uri, api_key, filepath):
    audio, sr = sf.read(filepath)
    assert sr == 16000, f"Sample rate must be 16kHz, got {sr}"
    if audio.ndim > 1:  # convert to mono
        audio = audio.mean(axis=1)
    # Convert float32 [-1,1] to int16
    if audio.dtype != np.int16:
        audio = (audio * 32767).astype(np.int16)
    chunk_size = 3200  # 100ms chunks at 16kHz mono
    async with websockets.connect(uri, extra_headers={"X-API-Key": api_key}) as ws:
        for i in range(0, len(audio), chunk_size):
            chunk = audio[i:i+chunk_size].tobytes()
            b64 = base64.b64encode(chunk).decode("ascii")
            await ws.send(json.dumps({"type": "input_audio_buffer.append", "audio": b64}))
        await ws.send(json.dumps({"type": "input_audio_buffer.commit"}))
        await ws.send(json.dumps({"type": "input_audio_buffer.flush"}))
        # Read server response
        while True:
            msg = await ws.recv()
            print(msg)

asyncio.run(send_audio(
    "ws://localhost:8080/ws/realtime",
    "sk_live_3jf83jf83jf",
    "output.pcm"
))

Task exception was never retrieved
future: <Task finished name='Task-6' coro=<main() done, defined at /tmp/ipykernel_82038/28440973.py:8> exception=ConnectionClosedOK(Close(code=1000, reason=''), Close(code=1000, reason=''), True)>
Traceback (most recent call last):
  File "/home/proxima/anaconda3/lib/python3.13/asyncio/tasks.py", line 306, in __step_run_and_handle_result
    result = coro.throw(exc)
  File "/tmp/ipykernel_82038/28440973.py", line 18, in main
    msg = await ws.recv()
          ^^^^^^^^^^^^^^^
  File "/home/proxima/PROXIMA-AI/voice_assistant/.venv/lib/python3.13/site-packages/websockets/asyncio/connection.py", line 324, in recv
    raise self.protocol.close_exc from self.recv_exc
websockets.exceptions.ConnectionClosedOK: received 1000 (OK); then sent 1000 (OK)


LibsndfileError: Error opening 'output.pcm': System error.